In [ ]:
import pandas as pd
import statsmodels.formula.api as smf

experiments = [
    {'name': 'TwoBandit', 'experiment': 'exp1', 'num_options': 2, 'held-out': False},
    {'name': 'TwoBandit', 'experiment': 'exp2', 'num_options': 2, 'held-out': False},
    {'name': 'DriftingBandit', 'experiment': 'exp0', 'num_options': 4, 'held-out': False},
    {'name': 'HorizonSomer', 'experiment': 'exp0', 'num_options': 2, 'held-out': False},
    {'name': 'HorizonWaltz', 'experiment': 'exp0', 'num_options': 2, 'held-out': False},
    {'name': 'HorizonSade', 'experiment': 'exp0', 'num_options': 2, 'held-out': True},
    {'name': 'HorizonFeng', 'experiment': 'exp0', 'num_options': 2, 'held-out': True},
    {'name': 'ChangingBandit', 'experiment': 'exp0', 'num_options': 2, 'held-out': True},
    {'name': 'MaggiesFarm', 'experiment': 'exp0', 'num_options': 3, 'held-out': True},
]

MODEL_NAMES = {
    "LLMPredict/Base/": "Qwen3-Coder-Next - Base",
    "LLMPredict/FineTuned/": "Qwen3-Coder-Next - Fine-Tuned",
    "RescorlaWagnerResults/": "Rescorla-Wagner",
    "EvolvedCogModel/Base/": "OpenEvolve Model - Base",
    "EvolvedCogModel/FineTuned/": "OpenEvolve Model - Fine-Tuned",
}

data_paths = ['LLMPredict/Base/', 'LLMPredict/FineTuned/', 'RescorlaWagnerResults/', 'EvolvedCogModel/FineTuned/', 'EvolvedCogModel/Base/']

repo_url = (
    "https://huggingface.co/datasets/"
    "cObliUvA/llm-guided-modelling-results/resolve/main"
)

### Descriptive Statistics
Computing mean negative log-likelihoods and standard deviations

In [ ]:
# ----------- LOAD EXPERIMENT-LEVEL MEAN NLL -----------
rows = []

for path in data_paths:

    name = MODEL_NAMES[path]

    for exp in experiments:

        exp_id = f"{exp['name']}_{exp['experiment']}"

        df = pd.read_csv(
            f"{repo_url}/{path}{exp_id}.csv"
        )

        rows.append({
            "model": name,
            "experiment": exp_id,
            "held_out": exp["held-out"],

            # Mean NLL within this experiment
            "mean_nll": df["nll"].mean()
        })


experiment_means = pd.DataFrame(rows)


# ----------- REPORT EXPERIMENT-WEIGHTED MEAN AND SD -----------
for model in MODEL_NAMES.values():

    df = experiment_means[
        experiment_means["model"] == model
    ]

    included = df[~df["held_out"]]["mean_nll"]
    held_out = df[df["held_out"]]["mean_nll"]
    overall = df["mean_nll"]

    print("\n" + "=" * 60)
    print(model)
    print("=" * 60)

    print(
        f"Included: "
        f"M = {included.mean():.3f}, "
        f"SD = {included.std():.3f}"
    )

    print(
        f"Held-out: "
        f"M = {held_out.mean():.3f}, "
        f"SD = {held_out.std():.3f}"
    )

    print(
        f"Overall:  "
        f"M = {overall.mean():.3f}, "
        f"SD = {overall.std():.3f}"
    )

### Inferential Statistics H1

In [ ]:
rows = []

for exp in experiments:

    exp_id = f"{exp['name']}_{exp['experiment']}"

    # ----------- LOAD CCM & RW -----------
    dfs = {}

    for model_name, folder in [
        ("CCM", "EvolvedCogModel/Base/"),
        ("RW", "RescorlaWagnerResults/")
    ]:
        df = pd.read_csv(f"{repo_url}/{folder}{exp_id}.csv")

        sort_cols = ["participant"]

        if "game" in df.columns:
            sort_cols.append("game")

        if "trial" in df.columns:
            sort_cols.append("trial")

        df = df.sort_values(sort_cols)

        # Keep identifiers + NLL
        id_cols = ["participant", "nll"]

        if "game" in df.columns:
            id_cols.append("game")

        if "trial" in df.columns:
            id_cols.append("trial")

        df = df[id_cols].copy()

        # Rename NLL according to model
        df = df.rename(columns={"nll": f"nll_{model_name}"})

        dfs[model_name] = df

    # ----------- ALIGN CCM & RW -----------
    merge_cols = ["participant"]

    if "game" in dfs["CCM"].columns:
        merge_cols.append("game")

    if "trial" in dfs["CCM"].columns:
        merge_cols.append("trial")

    paired = pd.merge(
        dfs["CCM"],
        dfs["RW"],
        on=merge_cols,
        how="inner"
    )

    # ----------- COMPUTE PAIRED DIFFERENCE -----------
    # RW - CCM
    # Negative values indicate that the RW model has lower NLL.
    # Positive values indicate that the CCM has lower NLL.

    paired["delta_nll"] = (
        paired["nll_CCM"] -
        paired["nll_RW"]
    )

    # Unique participant identifier across experiments
    paired["participant_uid"] = (
        exp_id + "_" + paired["participant"].astype(str)
    )

    paired["experiment"] = exp_id

    rows.append(
        paired[
            merge_cols
            + [
                "experiment",
                "participant_uid",
                "nll_CCM",
                "nll_RW",
                "delta_nll"
            ]
        ]
    )


# Combine all experiments
df_h1 = pd.concat(rows, ignore_index=True)


# ----------- H1 MODEL -----------
# Delta NLL ~ experiment

# The intercept represents the average RW - CCM difference for the reference coding.

# Negative values indicate better RW performance.
# Positive values indicate better CCM performance.

# Experiment effects allow the average difference to vary
# across experiments.

model = smf.ols(
    "delta_nll ~ C(experiment, Sum)",
    data=df_h1
).fit(
    cov_type="cluster",
    cov_kwds={"groups": df_h1["participant_uid"]}
)


print(model.summary())

print(f"\nN observations: {len(df_h1)}")
print(f"N participants: {df_h1['participant_uid'].nunique()}")
print(f"N experiments:  {df_h1['experiment'].nunique()}")

print("\nMean ΔNLL by experiment:")
print(
    df_h1
    .groupby("experiment")["delta_nll"]
    .agg(["mean", "std", "count"])
)

print("\nOverall mean ΔNLL:")
print(df_h1["delta_nll"].mean())

print("\nMean NLL by model:")
print(
    df_h1[["nll_CCM", "nll_RW"]]
    .mean()
)

### Inferential Statistics H3

In [ ]:
rows = []

for exp in experiments:

    exp_id = f"{exp['name']}_{exp['experiment']}"

    # ----------- LOAD CCMs -----------
    dfs = {}

    for cog, folder in [
        ("Base", "EvolvedCogModel/Base/"),
        ("FineTuned", "EvolvedCogModel/FineTuned/")
    ]:
        df = pd.read_csv(f"{repo_url}/{folder}{exp_id}.csv")

        sort_cols = ["participant"]

        if "game" in df.columns:
            sort_cols.append("game")

        if "trial" in df.columns:
            sort_cols.append("trial")

        df = df.sort_values(sort_cols)

        # Keep identifiers + NLL
        id_cols = ["participant", "nll"]

        if "game" in df.columns:
            id_cols.append("game")

        if "trial" in df.columns:
            id_cols.append("trial")

        df = df[id_cols].copy()

        # Rename NLL according to model
        df = df.rename(columns={"nll": f"nll_{cog}"})

        dfs[cog] = df

    # ----------- ALIGN CCMs -----------
    merge_cols = ["participant"]

    if "game" in dfs["Base"].columns:
        merge_cols.append("game")

    if "trial" in dfs["Base"].columns:
        merge_cols.append("trial")

    paired = pd.merge(
        dfs["Base"],
        dfs["FineTuned"],
        on=merge_cols,
        how="inner"
    )

    # ----------- CALCULATE PAIRED DIFFERENCE -----------
    paired["delta_nll"] = (
        paired["nll_FineTuned"] -
        paired["nll_Base"]
    )

    # Unique participant identifier across experiments
    paired["participant_uid"] = (
        exp_id + "_" + paired["participant"].astype(str)
    )

    paired["experiment"] = exp_id

    rows.append(
        paired[
            merge_cols
            + ["experiment", "participant_uid",
               "nll_Base", "nll_FineTuned", "delta_nll"]
        ]
    )


# Combine all experiments
df_h3 = pd.concat(rows, ignore_index=True)


# ----------- H3 MODEL -----------
# Delta NLL ~ experiment
#
# The coefficient/intercept represents the average FineTuned - Base
# difference. Negative values indicate better performance of the
# FineTuned-generated CCM.

model = smf.ols(
    "delta_nll ~ C(experiment, Sum)",
    data=df_h3
).fit(
    cov_type="cluster",
    cov_kwds={"groups": df_h3["participant_uid"]}
)


print(model.summary())

print(f"\nN observations: {len(df_h3)}")
print(f"N participants: {df_h3['participant_uid'].nunique()}")
print(f"N experiments:  {df_h3['experiment'].nunique()}")

print("\nMean ΔNLL by experiment:")
print(
    df_h3
    .groupby("experiment")["delta_nll"]
    .agg(["mean", "std", "count"])
)

print("\nOverall mean ΔNLL:")
print(df_h3["delta_nll"].mean())